In [5]:
import pandas as pd
import numpy as np
from scipy.stats.contingency import association
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors

SEED = 42
np.random.seed(SEED)

# ── FILE PATHS ────────────────────────────────────────────────────────────
REAL_FILE   = "DS1.csv"     # update if your filename differs
SYNTH_FILE  = "OUTPUT.xlsx"   # update if your filename differs

# ── LOAD DATA ─────────────────────────────────────────────────────────────
real  = pd.read_csv(REAL_FILE)
synth = pd.read_excel(SYNTH_FILE)
synth.columns = real.columns    # align column names

print(f"Real  data : {real.shape}")
print(f"Synth data : {synth.shape}")

# ── LABEL ENCODE (union of real + synthetic categories) ───────────────────
real_enc, synth_enc = real.copy(), synth.copy()
for col in real.columns:
    le = LabelEncoder()
    le.fit(pd.concat([real_enc[col].astype(str),
                      synth_enc[col].astype(str)]))
    real_enc[col]  = le.transform(real_enc[col].astype(str))
    synth_enc[col] = le.transform(synth_enc[col].astype(str))

# ══════════════════════════════════════════════════════════════════════════
#  FIDELITY
# ══════════════════════════════════════════════════════════════════════════




# ── 1. Pearson correlation matrix L2 ─────────────────────────────────────
pearson_l2 = round(
    float(np.linalg.norm(real_enc.corr().values - synth_enc.corr().values)), 4
)



# ══════════════════════════════════════════════════════════════════════════
#  UTILITY — TSTR (Train-on-Synthetic Test-on-Real)
# ══════════════════════════════════════════════════════════════════════════
target  = real_enc.columns[-1]   # Q11. Plan to Shop Again in Future

X_real  = real_enc.drop(columns=[target]);  y_real  = real_enc[target]
X_synth = synth_enc.drop(columns=[target]); y_synth = synth_enc[target]

_, X_test, _, y_test = train_test_split(
    X_real, y_real, test_size=0.3, random_state=SEED
)

rf      = RandomForestClassifier(random_state=SEED).fit(X_synth, y_synth)
y_pred  = rf.predict(X_test)

tstr_acc = round(float(accuracy_score(y_test, y_pred)), 4)


# ══════════════════════════════════════════════════════════════════════════
#  PRIVACY — DCR (Distance to Closest Record)
# ══════════════════════════════════════════════════════════════════════════
X_train_real, _, _, _ = train_test_split(
    real_enc, real_enc[target], test_size=0.3, random_state=SEED
)
nn  = NearestNeighbors(n_neighbors=1).fit(X_train_real)
dcr = round(float(nn.kneighbors(synth_enc)[0].mean()), 4)

# ══════════════════════════════════════════════════════════════════════════
#  RESULTS SUMMARY
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "="*55)
print(f"  EVALUATION RESULTS — Experiment 1 — LLM Baseline")
print("="*55)
print(f"  FIDELITY")
print(f"    Pearson L2          : {pearson_l2}")
print(f"  UTILITY")
print(f"    TSTR Accuracy       : {tstr_acc}")

print(f"  PRIVACY")
print(f"    DCR (synth→train)   : {dcr}")
print("="*55)

# ── Save per-column table to Excel ────────────────────────────────────────
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side


df_agg = pd.DataFrame([
    {"Metric": "Pearson L2 (lower better)",       "Value": pearson_l2, "Direction": "Lower"},
    {"Metric": "TSTR Accuracy (higher better)",   "Value": tstr_acc,   "Direction": "Higher"},
    {"Metric": "DCR (higher = more privacy)",     "Value": dcr,        "Direction": "Higher"},
])


out_file = "Evaluation_E1.xlsx"
with pd.ExcelWriter(out_file) as writer:
    df_agg.to_excel(writer, sheet_name="Aggregate Metrics", index=False)


print(f"\nResults saved to: {out_file}")

Real  data : (272, 11)
Synth data : (272, 11)

  EVALUATION RESULTS — Experiment 1 — LLM Baseline
  FIDELITY
    Pearson L2          : 2.3621
  UTILITY
    TSTR Accuracy       : 0.5976
  PRIVACY
    DCR (synth→train)   : 2.4889

Results saved to: Evaluation_E1.xlsx
